In [32]:
import pandas as pd
import string
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.corpus import stopwords
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity


In [42]:
# Reading and cleaning the file
df = pd.read_csv('tmdb_5000_movies.csv', encoding=('ISO-8859-1'))
df = df.dropna(subset=['overview'])
df['title'] = df['title'].str.lower()
rows = df.iloc[0]
rows

budget                                                          237000000
genres                  [{"id": 28, "name": "Action"}, {"id": 12, "nam...
homepage                                      http://www.avatarmovie.com/
id                                                                  19995
keywords                [{"id": 1463, "name": "culture clash"}, {"id":...
original_language                                                      en
original_title                                                     Avatar
overview                In the 22nd century, a paraplegic Marine is di...
popularity                                                     150.437577
production_companies    [{"name": "Ingenious Film Partners", "id": 289...
production_countries    [{"iso_3166_1": "US", "name": "United States o...
release_date                                                   2009-12-10
revenue                                                        2787965087
runtime                               

In [34]:
# TFIDF Matrix
df['overview'] = df['overview'].str.lower()
vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = vectorizer.fit_transform(df['overview'])

In [35]:
# Applying LSA
svd = TruncatedSVD(n_components=100, random_state=42)
lsa_matrix = svd.fit_transform(tfidf_matrix)

In [44]:
def recommend(matrix, movie_name):
    similarity_matrix = cosine_similarity(matrix, matrix)
    i = df[df['title'] == movie_name].index[0]
    similarity_scores = list(enumerate(similarity_matrix[i]))
    similarity_scores = sorted(similarity_scores, key=lambda x: x[1], reverse=True)
    similarity_scores = similarity_scores[1:6]
    movie_indices = [j[0] for j in similarity_scores]
    return df['title'].iloc[movie_indices]

In [46]:
while True:
    movie = input("what is one movie you enjoyed:\n").lower()

    if not df[df['title'] == movie].empty:
        break
    else:
        print("write a valid movie name")

# LSA results
print("LSA results\n", recommend(lsa_matrix, movie))

# TFIDF results
print("TFIDF results\n", recommend(tfidf_matrix, movie))

LSA results
 66                     up
1248       at first sight
3972     chicago overcoat
3134            partition
3788    guten tag, ramã³n
Name: title, dtype: object
TFIDF results
 2897                                cypher
134     mission: impossible - rogue nation
1930                            stone cold
914                   central intelligence
1683                       pitch perfect 2
Name: title, dtype: object
